## Imports


#### Step 1: Upload CSV Files Manually

Before running this notebook, **upload the following CSV files** from your local computer:

- `farm_type.csv`
- `herd.csv`
- `herd_type.csv`

These files should be located in:  
`eufmdis (eufmdis home folder)\data\hungary_demo`

#### How to Upload Files in Google Colab?

1. On the left-hand side of the Colab interface, click the **Files** tab (the folder icon just below the key icon).
2. In the Files pane, click the **Upload to session storage** button (the icon with an upwards pointing arrow; it should be the first icon in the pane).
3. In the file dialog that appears, select the three CSV files: `farm_type.csv`, `herd.csv`, and `herd_type.csv`.
4. Verify that these files appear in the Files pane on the left.

Once you have verified that the files are uploaded, run the cell below by clicking the play button (located at the top left corner of the cell). If you don't see the play button, hover over the cell to reveal it.


## The Model

The model computes spatial risk as the product of two surfaces:

$$
I(x) = \varphi\left(\sum_i w_i \, \exp\left(-\frac{d(x, x_i)^2}{2\sigma^2}\right)\right)
$$

$$
S(x) = \varphi\left(\sum_i w_s \, \exp\left(-\frac{d(x, x_i)^2}{2\sigma^2}\right)\right)
$$

$$
\text{risk}(x) = I(x) \cdot S(x)
$$


### Infectivity Model

The weight used for infectivity is computed as:

$$
w_i = \frac{S_i \, n^{P_i}}{\mathrm{population\_mean}\Bigl(S_i \, n^{P_i}\Bigr)}
$$

**Where:**
- $w_i$: Infectivity weight (for species \(i\))
- $S_i$: Species-relative infectivity
- $n$: Herd size (the number of individuals in the group)
- $P_i$: Species infectivity power
- $\varphi$: Is the normalizing function, it could be:
    - *Linear*  $\varphi(x) = x$
    - *Max div* $\varphi(x) = \frac{x}{\max(x)}$
    - *Sigmoid* $\varphi(x) = \frac{1}{1 + e^{-x}}$
    - *Exponential* $\varphi(x) = 1 - e^{-x}$

These parameters are provided by the `herd_type.csv` file.

### Susceptibility Model

Similarly, the susceptibility weight is computed as:

$$
w_s = \frac{S_s \, n^{P_s}}{\mathrm{population\_mean}\Bigl(S_s \, n^{P_s}\Bigr)}
$$

**Where:**
- $w_s$: Susceptibility weight (for species \(i\))
- $S_s$: Species-relative susceptibility
- $n$: Herd size (the number of individuals in the group)
- $P_s$: Species susceptibility power
- $\varphi$: Is the normalizing function, it could be:
    - *Linear*  $\varphi(x) = x$
    - *Max div* $\varphi(x) = \frac{x}{\max(x)}$
    - *Sigmoid* $\varphi(x) = \frac{1}{1 + e^{-x}}$
    - *Exponential* $\varphi(x) = 1 - e^{-x}$


**Dont forget to run the cell below** to be able to use the interactive controls

In [ ]:

from matplotlib import pyplot as plt
%matplotlib inline

from functools import lru_cache
from IPython.display import display, clear_output
from ipywidgets import widgets
import scipy.ndimage
import scipy.signal
import numpy as np
import pandas as pd
import pyproj
from pathlib import Path
from shapely.geometry import shape
from shapely.ops import transform, unary_union
from shapely import contains_xy
import fiona
import cartopy.crs as ccrs


def resolve_data_path(filename: str) -> Path:
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / 'python' / 'map' / filename,
        Path.cwd() / 'map' / filename,
        Path('/content') / filename,
    ]
    if '__file__' in globals():
        notebook_dir = Path(__file__).resolve().parent
        candidates.insert(0, notebook_dir / filename)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    searched = '\n'.join(str(path.resolve()) for path in candidates)
    raise FileNotFoundError(f'Could not find {filename}. Searched:\n{searched}')


def load_country_geometry(shapefile_path: str):
    shapefile_path = resolve_data_path(shapefile_path)
    county_geometries = []
    with fiona.open(str(shapefile_path), encoding='iso-8859-1') as shp:
        original_crs = shp.crs or 'EPSG:23700'
        for rec in shp:
            geom = shape(rec['geometry'])
            county_geometries.append(geom)
    country = unary_union(county_geometries)
    proj = pyproj.Transformer.from_crs(original_crs, 'EPSG:3857', always_xy=True).transform
    return transform(proj, country)


def load_herds(herd_csv_path: str, herd_type_csv_path: str, farm_type_csv_path: str):
    herd_df = pd.read_csv(resolve_data_path(herd_csv_path), header=0)
    herd_type_df = pd.read_csv(resolve_data_path(herd_type_csv_path), header=0).rename(
        columns={'herd type id': 'herd type'})
    farm_type_df = pd.read_csv(resolve_data_path(farm_type_csv_path), header=0).rename(columns={'id': 'farm type'})

    combined_df = pd.merge(herd_df, herd_type_df, on='herd type')
    combined_df = pd.merge(combined_df, farm_type_df, on='farm type')

    lon_col, lat_col = 'herd long', 'herd lat'
    transformer = pyproj.Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)
    xs, ys = transformer.transform(combined_df[lon_col].values, combined_df[lat_col].values)
    combined_df['x'] = xs
    combined_df['y'] = ys
    return combined_df


def make_grid(geometry):
    minx, miny, maxx, maxy = geometry.bounds
    buffer = 5000  # Extend the grid 5km beyond the country boundary
    xi = np.linspace(minx - buffer, maxx + buffer, 1000)
    yi = np.linspace(miny - buffer, maxy + buffer, 1000)
    x_centers = (xi[:-1] + xi[1:]) / 2
    y_centers = (yi[:-1] + yi[1:]) / 2
    grid_x, grid_y = np.meshgrid(x_centers, y_centers, indexing='ij')
    return xi, yi, grid_x, grid_y


get_norm_func = {
    'linear': lambda x: x,
    'div_max': lambda x: x / np.nanmax(x) if np.nanmax(x) != 0 else x,
    'sigmoid': lambda x: 1 / (1 + np.exp(-x)),
    'one_minus_exp': lambda x: 1 - np.exp(-x)
}


@lru_cache(maxsize=16)
def make_power_law_kernel(nx, ny, dx_km, dy_km, a_km, gamma):
    x_offsets = np.arange(-nx + 1, nx) * dx_km
    y_offsets = np.arange(-ny + 1, ny) * dy_km
    xx, yy = np.meshgrid(x_offsets, y_offsets, indexing='ij')
    distances = np.sqrt(xx ** 2 + yy ** 2)
    kernel = 1.0 / np.power(distances + a_km, gamma)
    kernel /= kernel.sum()
    return kernel


def apply_spatial_kernel(hist, kernel_name, sigma_pixels, dx_km, dy_km, a_km, gamma):
    if kernel_name == 'gaussian':
        return scipy.ndimage.gaussian_filter(hist, sigma=sigma_pixels)
    if kernel_name == 'power_law':
        kernel = make_power_law_kernel(
            hist.shape[0], hist.shape[1], round(dx_km, 6), round(dy_km, 6), round(a_km, 6), round(gamma, 6)
        )
        return scipy.signal.fftconvolve(hist, kernel, mode='same')
    raise ValueError(f'Unknown kernel: {kernel_name}')


def compute_heatmap(sigma_km, time_horizon, kernel_name, a_km, gamma, xi, yi, df, mask, user_params, norm_func,
                    normalize_by_selected_means):
    sigma_m = sigma_km * 1000.0
    dx = xi[1] - xi[0]
    dy = yi[1] - yi[0]
    sigma_pixels = sigma_m / dx
    dx_km = dx / 1000.0
    dy_km = dy / 1000.0

    selected_types = list(user_params.keys())
    df_filtered = df[df['herd type name'].isin(selected_types)].copy()
    if df_filtered.empty:
        empty_grid = np.zeros((len(xi) - 1, len(yi) - 1))
        empty_grid[~mask] = np.nan
        return empty_grid.copy(), empty_grid.copy(), empty_grid.copy(), empty_grid.copy()

    for ht, params in user_params.items():
        mask_ht = (df_filtered['herd type name'] == ht)
        df_filtered.loc[mask_ht, 'raw infectivity'] = params['inf_weight'] * (
                df_filtered.loc[mask_ht, 'herd size'] ** params['inf_power']
        )
        df_filtered.loc[mask_ht, 'raw susceptibility'] = params['susc_weight'] * (
                df_filtered.loc[mask_ht, 'herd size'] ** params['susc_power']
        )

    if normalize_by_selected_means:
        infectivity_mean = df_filtered['raw infectivity'].mean()
        susceptibility_mean = df_filtered['raw susceptibility'].mean()
        df_filtered['infectivity'] = (
            df_filtered['raw infectivity'] / infectivity_mean if infectivity_mean != 0 else df_filtered[
                'raw infectivity']
        )
        df_filtered['susceptibility'] = (
            df_filtered['raw susceptibility'] / susceptibility_mean
            if susceptibility_mean != 0 else df_filtered['raw susceptibility']
        )
    else:
        df_filtered['infectivity'] = df_filtered['raw infectivity']
        df_filtered['susceptibility'] = df_filtered['raw susceptibility']

    xs, ys = df_filtered['x'].values, df_filtered['y'].values

    # Create weighted 2D histograms.
    I_hist, _, _ = np.histogram2d(xs, ys, bins=[xi, yi], weights=df_filtered['infectivity'])
    S_hist, _, _ = np.histogram2d(xs, ys, bins=[xi, yi], weights=df_filtered['susceptibility'])

    I_grid = norm_func(apply_spatial_kernel(I_hist, kernel_name, sigma_pixels, dx_km, dy_km, a_km, gamma))
    S_grid = norm_func(apply_spatial_kernel(S_hist, kernel_name, sigma_pixels, dx_km, dy_km, a_km, gamma))
    risk_grid = I_grid * S_grid
    infection_prob_grid = 1 - np.exp(-time_horizon * np.nan_to_num(risk_grid, nan=0.0))

    I_grid[~mask] = np.nan
    S_grid[~mask] = np.nan
    risk_grid[~mask] = np.nan
    infection_prob_grid[~mask] = np.nan

    return I_grid, S_grid, risk_grid, infection_prob_grid


def plot_heatmap(kernel_name, sigma_km, time_horizon, a_km, gamma, xi, yi, grid_x, grid_y, geometry, I_grid, S_grid,
                 risk_grid,
                 infection_prob_grid, uniform):
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4), dpi=150,
                             subplot_kw={'projection': ccrs.epsg(3857)})
    titles = [
        'Infectivity Surface (I)',
        'Susceptibility Surface (S)',
        'Risk Surface (I * S)',
        'Infection Probability (1 - exp(-t * lambda))'
    ]
    grids = [I_grid, S_grid, risk_grid, infection_prob_grid]

    for ax, grid, title in zip(axes, grids, titles):
        ax.set_extent([xi.min(), xi.max(), yi.min(), yi.max()], crs=ccrs.epsg(3857))
        ax.set_facecolor("white")
        for spine in ax.spines.values():
            spine.set_visible(False)
        if uniform:
            im = ax.pcolormesh(grid_x, grid_y, grid, transform=ccrs.epsg(3857),
                               cmap='turbo', shading='auto', alpha=0.8, vmin=0, vmax=1)
        else:
            im = ax.pcolormesh(grid_x, grid_y, grid, transform=ccrs.epsg(3857),
                               cmap='turbo', shading='auto', alpha=0.8)
        fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.046, pad=0.04)
        ax.add_geometries([geometry], crs=ccrs.epsg(3857),
                          facecolor='none', edgecolor='black', linewidth=1.5, zorder=3)
        ax.set_title(title, fontsize=14)
    kernel_label = (
        f'Gaussian, sigma: {sigma_km:.2f} km' if kernel_name == 'gaussian'
        else f'Power law, a: {a_km:.2f} km, gamma: {gamma:.2f}'
    )
    plt.suptitle(f'Heat Maps ({kernel_label}, t: {time_horizon:.2f})', fontsize=16)
    plt.tight_layout()
    return fig


def create_controls(defaults, update_callback):
    def make_weight_slider(default, label):
        return widgets.FloatSlider(
            value=default, min=0.1, max=10, step=0.1, description=label,
            continuous_update=False, layout=widgets.Layout(width='250px')
        )

    def make_power_slider(default, label):
        return widgets.FloatSlider(
            value=default, min=0, max=1, step=0.05, description=label,
            continuous_update=False, layout=widgets.Layout(width='250px')
        )

    norm_dropdown = widgets.Dropdown(
        options=[
            ("Linear", "linear"),
            ("Max div", "div_max"),
            ("Sigmoid", "sigmoid"),
            ("Exponential", "one_minus_exp")
        ],
        value='linear',
        description='Normalizing function:',
        layout=widgets.Layout(width='400px')
    )

    norm_formula_widget = widgets.HTMLMath(value=r'$\varphi(x)=x$')

    def update_norm_formula(change):
        mapping = {
            "linear": r'$\varphi(x)=x$',
            "div_max": r'$\varphi(x)=\frac{x}{\max(x)}$',
            "sigmoid": r'$\varphi(x)=\frac{1}{1+e^{-x}}$',
            "one_minus_exp": r'$\varphi(x)=1-e^{-x}$'
        }
        norm_formula_widget.value = mapping.get(change['new'], r'$\varphi(x)=1-e^{-x}$')
        on_value_change(change)  # force replotting

    norm_dropdown.observe(update_norm_formula, names='value')

    distance_slider = widgets.FloatSlider(
        value=4.0, min=0.1, max=40.0, step=0.2, description='Sigma (km):',
        continuous_update=False, layout=widgets.Layout(width='300px')
    )

    kernel_dropdown = widgets.Dropdown(
        options=[('Gaussian', 'gaussian'), ('Power law', 'power_law')],
        value='gaussian',
        description='Kernel:',
        layout=widgets.Layout(width='250px')
    )

    kernel_formula_widget = widgets.HTMLMath(value=r'$K(d)=\exp\left(-\frac{d^2}{2\sigma^2}\right)$')

    gamma_slider = widgets.FloatSlider(
        value=2.0, min=0.5, max=5.0, step=0.1, description='gamma:',
        continuous_update=False, layout=widgets.Layout(width='300px')
    )

    time_slider = widgets.FloatSlider(
        value=14.0, min=0.1, max=100.0, step=0.1, description='Time (t):',
        continuous_update=False, layout=widgets.Layout(width='300px')
    )

    def get_user_params():
        user_params = {}
        for ht, controls in herd_controls.items():
            if controls['checkbox'].value:
                user_params[ht] = {
                    'inf_weight': controls['inf_weight'].value,
                    'inf_power': controls['inf_power'].value,
                    'susc_weight': controls['susc_weight'].value,
                    'susc_power': controls['susc_power'].value,
                }
        return user_params

    def on_value_change(change):
        distance_val = distance_slider.value
        time_val = time_slider.value
        kernel_val = kernel_dropdown.value
        gamma_val = gamma_slider.value
        params = get_user_params()
        norm_func = get_norm_func[norm_dropdown.value]
        normalize_by_selected_means_val = normalize_by_selected_means_checkbox.value
        uniform_val = uniform_checkbox.value
        out.clear_output(wait=True)
        with out:
            display(widgets.HTML("<h3>Computing... please wait</h3>"))
        update_callback(
            distance_val, time_val, kernel_val, distance_val, gamma_val, params, norm_func,
            normalize_by_selected_means_val, uniform_val
        )

    def update_kernel_controls(change=None):
        is_gaussian = kernel_dropdown.value == 'gaussian'
        distance_slider.description = 'Sigma (km):' if is_gaussian else 'a (km):'
        gamma_slider.disabled = is_gaussian
        if is_gaussian:
            kernel_formula_widget.value = r'$K(d)=\exp\left(-\frac{d^2}{2\sigma^2}\right)$'
        else:
            kernel_formula_widget.value = r'$K(d)=\frac{1}{(d+a)^\gamma}$'
        if change is not None:
            on_value_change(change)

    normalize_by_selected_means_checkbox = widgets.Checkbox(
        value=True,
        description="Normalize by selected herd means",
        layout=widgets.Layout(width='280px')
    )
    normalize_by_selected_means_checkbox.observe(on_value_change, names='value')

    uniform_checkbox = widgets.Checkbox(value=False, description="Uniform color", layout=widgets.Layout(width='200px'))
    uniform_checkbox.observe(on_value_change, names='value')

    herd_controls = {}
    herd_controls_rows = []
    for ht in defaults.keys():
        chk = widgets.Checkbox(value=(False if ht.lower() == 'backyard' else True), description=ht,
                               layout=widgets.Layout(width='350px'))
        inf_weight_slider = make_weight_slider(defaults[ht]['inf_weight'], 'Inf Wt:')
        inf_power_slider = make_power_slider(defaults[ht]['inf_power'], 'Inf Pow:')
        susc_weight_slider = make_weight_slider(defaults[ht]['susc_weight'], 'Susc Wt:')
        susc_power_slider = make_power_slider(defaults[ht]['susc_power'], 'Susc Pow:')
        reset_btn = widgets.Button(description='Reset', layout=widgets.Layout(width='60px'))

        herd_controls[ht] = {
            'checkbox': chk,
            'inf_weight': inf_weight_slider,
            'inf_power': inf_power_slider,
            'susc_weight': susc_weight_slider,
            'susc_power': susc_power_slider,
            'reset_button': reset_btn
        }
        row = widgets.HBox([chk, inf_weight_slider, inf_power_slider,
                            susc_weight_slider, susc_power_slider, reset_btn])
        herd_controls_rows.append(row)

    control_panel = widgets.VBox(
        [widgets.HBox([kernel_dropdown, kernel_formula_widget]),
         widgets.HBox([distance_slider, gamma_slider, time_slider]),
         widgets.HBox([
             norm_dropdown, norm_formula_widget, normalize_by_selected_means_checkbox, uniform_checkbox
         ])] + herd_controls_rows)

    def on_checkbox_change(change, ht):
        controls = herd_controls[ht]
        new_state = change['new']
        controls['inf_weight'].disabled = not new_state
        controls['inf_power'].disabled = not new_state
        controls['susc_weight'].disabled = not new_state
        controls['susc_power'].disabled = not new_state
        on_value_change(change)

    def on_reset_clicked(b, ht):
        controls = herd_controls[ht]
        defaults_for_ht = defaults[ht]
        controls['inf_weight'].value = defaults_for_ht['inf_weight']
        controls['inf_power'].value = defaults_for_ht['inf_power']
        controls['susc_weight'].value = defaults_for_ht['susc_weight']
        controls['susc_power'].value = defaults_for_ht['susc_power']
        on_value_change(None)

    distance_slider.observe(on_value_change, names='value')
    kernel_dropdown.observe(update_kernel_controls, names='value')
    gamma_slider.observe(on_value_change, names='value')
    time_slider.observe(on_value_change, names='value')
    norm_dropdown.observe(on_value_change, names='value')
    for ht, controls in herd_controls.items():
        controls['checkbox'].observe(lambda change, ht=ht: on_checkbox_change(change, ht), names='value')
        controls['inf_weight'].observe(on_value_change, names='value')
        controls['inf_power'].observe(on_value_change, names='value')
        controls['susc_weight'].observe(on_value_change, names='value')
        controls['susc_power'].observe(on_value_change, names='value')
        controls['reset_button'].on_click(lambda b, ht=ht: on_reset_clicked(b, ht))

    update_kernel_controls()
    return (
        control_panel,
        kernel_dropdown,
        distance_slider,
        gamma_slider,
        time_slider,
        herd_controls,
        norm_dropdown,
        normalize_by_selected_means_checkbox,
    )


def update_plot(sigma_km, time_horizon, kernel_name, a_km, gamma, user_params, xi, yi, grid_x, grid_y, geometry,
                herd_df, mask, out,
                norm_func, normalize_by_selected_means, uniform):
    I_grid, S_grid, risk_grid, infection_prob_grid = compute_heatmap(
        sigma_km, time_horizon, kernel_name, a_km, gamma, xi, yi, herd_df, mask, user_params, norm_func,
        normalize_by_selected_means
    )
    fig = plot_heatmap(
        kernel_name, sigma_km, time_horizon, a_km, gamma, xi, yi, grid_x, grid_y, geometry, I_grid, S_grid, risk_grid,
        infection_prob_grid, uniform
    )
    with out:
        clear_output(wait=True)
        display(fig)
        plt.close(fig)


def main():
    geometry = load_country_geometry('megye.shp')
    herd_df = load_herds('herd.csv', 'herd_type.csv', 'farm_type.csv')
    xi, yi, grid_x, grid_y = make_grid(geometry)
    mask = contains_xy(geometry, grid_x, grid_y)

    defaults = {}
    for ht in sorted(herd_df['herd type name'].unique()):
        row = herd_df[herd_df['herd type name'] == ht].iloc[0]
        defaults[ht] = {
            'inf_weight': row['inf weight'],
            'inf_power': row['inf power'],
            'susc_weight': row['susc weight'],
            'susc_power': row['susc power']
        }

    global out
    out = widgets.Output()

    def sync_update(sigma_val, time_val, kernel_name, a_km, gamma, user_params, norm_func,
                    normalize_by_selected_means, uniform):
        update_plot(
            sigma_val, time_val, kernel_name, a_km, gamma, user_params, xi, yi, grid_x, grid_y, geometry, herd_df, mask,
            out,
            norm_func, normalize_by_selected_means, uniform
        )

    (
        control_panel,
        kernel_dropdown,
        distance_slider,
        gamma_slider,
        time_slider,
        herd_controls,
        norm_dropdown,
        normalize_by_selected_means_checkbox,
    ) = create_controls(
        defaults, sync_update)
    display(control_panel, out)

    def get_user_params():
        user_params = {}
        for ht, controls in herd_controls.items():
            if controls['checkbox'].value:
                user_params[ht] = {
                    'inf_weight': controls['inf_weight'].value,
                    'inf_power': controls['inf_power'].value,
                    'susc_weight': controls['susc_weight'].value,
                    'susc_power': controls['susc_power'].value,
                }
        return user_params

    sync_update(
        distance_slider.value, time_slider.value, kernel_dropdown.value, distance_slider.value, gamma_slider.value,
        get_user_params(), get_norm_func[norm_dropdown.value], normalize_by_selected_means_checkbox.value, False
    )


if __name__ == '__main__':
    main()


Output()